# Feature PDM

### Imports

In [16]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from ppvm_py.utils import load_object

from ppvm_py.plotting import plot_timeseries
from ppvm_py.plotting import plot_heatmap
from ppvm_py.plotting import generate_2d_video

from ppvm_py.data_processing.patt_fld8v import print_xyz_slice_example_from_parsed_data

from ppvm_py.data_processing.utils import create_index_to_coord_map

from ppvm_py.feature_generation.sdc import compute_symmetric_difference_coefficients_with_index_to_coord_map

### General Args

In [17]:
# IMPORTANT: Restart Kernel when swapping datasets to reload them!
Ha = 300

if Ha == 300:
    patt_fld8v_folder_path = Path("../../../../../data/4pi_re1000_ha300.360_ch28.p1/patt_fld8v/")
elif Ha == 1000:
    patt_fld8v_folder_path = Path("../../../../../data/4pi_re1000_ha1000.384.pi/patt_fld8v/")

fld_path = patt_fld8v_folder_path / Path("pkl/patt_fld8v.pkl")

# Estimates from evaluations
sigma = -1  # According to the results here.
B_z = -1  # According to the results here.

animation_folder = Path(f"../animations/feature_pdm/ha{Ha}/")

truncate_data = False
truncate_at_snapshot = 10  # Including indices 0 to (truncate_at_snapshot - 1)

test_size = 0.3

sdc_applicable_region = (
    slice(None),  # time
    slice(None),  # x
    slice(1, -1),  # y, sdc in y => boundaries in y are not applicable
    slice(None),  # z
)

### Preamble

In [18]:
mean_mse = {}  # Dict for gathering mean-squared-errors.

### Load Data

In [19]:
if 'fld_data' not in locals():
    fld_data = load_object(fld_path)
    if truncate_data:
        fld_data['timeseries'] = fld_data['timeseries'][:truncate_at_snapshot]
    fld_data['index_to_coord_map'] = create_index_to_coord_map(fld_data['coord_to_index_map'])
print_xyz_slice_example_from_parsed_data(fld_data)

N, n_x, n_y, n_z, n_v = fld_data['timeseries'].shape  # N = num snapshots

The 3D data looks as follows:
N (number of snapshots) = 100
n_x (grid depth) = 541, n_y (grid height) = 91, n_z (grid width) = 91, n_v (number of variables) = 8
Labels:
['vx', 'vy', 'vz', 'jx', 'jy', 'jz', 'P', 'F']
Preview of the first snapshot (first 2 x, first 2 y, first 2 z) with coordinates:
  (x=0.00, y=-1.00, z=-1.00): [ 0.000000e+00  0.000000e+00  0.000000e+00 -1.464999e-19  0.000000e+00
  0.000000e+00 -4.031829e-04  1.353023e+00]
  (x=0.00, y=-1.00, z=-0.99): [ 0.000000e+00  0.000000e+00  0.000000e+00  3.428746e-20  0.000000e+00
 -1.176754e-02 -4.036609e-04  1.353055e+00]
  (x=0.00, y=-0.99, z=-1.00): [ 0.000000e+00  0.000000e+00  0.000000e+00 -4.206331e-19 -4.652879e-03
  0.000000e+00 -4.037547e-04  1.353035e+00]
  (x=0.00, y=-0.99, z=-0.99): [ 0.000000e+00  0.000000e+00  0.000000e+00 -1.286717e-21 -4.781152e-03
 -1.184628e-02 -4.044691e-04  1.353067e+00]
  (x=0.02, y=-1.00, z=-1.00): [ 0.000000e+00  0.000000e+00  0.000000e+00  8.869907e-02  0.000000e+00
  0.000000e+00 -3.996

### Define Train and Test Set

In [20]:
fld_data_train, fld_data_test = train_test_split(
    fld_data['timeseries'],
    test_size=test_size,
    shuffle=False,
)
n_timesteps_train = fld_data_train.shape[0]
n_timesteps_test = fld_data_test.shape[0]

n_grid_points_no_y_boundary = n_x * (n_y - 2) * n_z

### Define $y_{true}$

In [21]:
y_true_train = fld_data_train[
    ...,
    fld_data['labels'].index('vx'),
][sdc_applicable_region]
y_true_train = y_true_train.reshape(n_timesteps_train, -1)

y_true_test = fld_data_test[
    ...,
    fld_data['labels'].index('vx'),
][sdc_applicable_region]
y_true_test = y_true_test.reshape(n_timesteps_test, -1)

# Every grid point is a feature. The scaling must not be different between these features => train SS on all data together.
ss_y_true_trainer = StandardScaler()
ss_y_true_trainer.fit(y_true_train.reshape(-1, 1))

# Create on SS that works on all features separately with the training params from the SS on all data.
ss_y_true = StandardScaler()

ss_y_true.scale_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.scale_[0],
)
ss_y_true.mean_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.mean_[0],
)
ss_y_true.var_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.var_[0],
)
ss_y_true.n_features_in_ = n_grid_points_no_y_boundary
ss_y_true.n_samples_seen_ = np.full(
    n_grid_points_no_y_boundary,
    ss_y_true_trainer.n_samples_seen_,
)

# trans y_true
y_true_train = ss_y_true.transform(
    y_true_train,
    copy=False,
    )
y_true_test = ss_y_true.transform(
    y_true_test,
    copy=False,
    )

### Precompute Features

In [22]:
approx_part_phi_y_train= compute_symmetric_difference_coefficients_with_index_to_coord_map(
    fld_data_train[..., fld_data['labels'].index('F')],
    direction_axis=2,
    index_to_coord_map=fld_data['index_to_coord_map'],
)
approx_part_phi_y_train = approx_part_phi_y_train.reshape(n_timesteps_train, -1)

approx_part_phi_y_test= compute_symmetric_difference_coefficients_with_index_to_coord_map(
    fld_data_test[..., fld_data['labels'].index('F')],
    direction_axis=2,
    index_to_coord_map=fld_data['index_to_coord_map'],
)
approx_part_phi_y_test = approx_part_phi_y_test.reshape(n_timesteps_test, -1)


v_x_pdm_train = approx_part_phi_y_train / B_z
v_x_pdm_train = ss_y_true.transform(
    v_x_pdm_train,
    copy=False,
)

v_x_pdm_test = approx_part_phi_y_test / B_z
v_x_pdm_test = ss_y_true.transform(
    v_x_pdm_test,
    copy=False,
)

j_y_test = fld_data_test[..., fld_data['labels'].index('jy')][sdc_applicable_region]
j_y_test = j_y_test.reshape(n_timesteps_test, -1)

### Potential Difference Method (SDC) Benchmark
Assume we know $j_y$, but we are restricted the the potential measurements $\phi$ from our DNS.

A simple feature used in the experiments is the symmetric difference coefficient $\frac{\Delta{\phi}}{\Delta{y}}$ to approximate $\frac{\partial{\phi}}{\partial{y}}$. This is also called *Potential Difference Method*. With this and $j_y$, Ohm's law can be approximated to calculate $v_x$.

We use the smallest difference possible to calculate $\frac{\Delta{\phi}}{\Delta{y}}$. With this, we have a benchmark for methods that use the Symmetric Difference Coefficient as singular feature.

##### Testing

In [23]:
v_x_sdc_test = (approx_part_phi_y_test - (j_y_test / sigma)) / B_z
y_pred_sdc_test = ss_y_true.transform(
    v_x_sdc_test,
    copy=False,
)

In [24]:
mse_sdc = mean_squared_error(
    y_true_test,
    y_pred_sdc_test,
    multioutput='raw_values',
)

mean_mse_sdc = np.mean(mse_sdc)
mean_mse['PDM Benchmark'] = float(mean_mse_sdc)
print(f"mean(mse(pdm_bench)) = {mean_mse_sdc}")

mean(mse(pdm_bench)) = 0.0038821085410341055


In [25]:
i_start = 0
generate_2d_video(
    data=mse_sdc.reshape(n_x, n_y - 2, n_z),
    output_file_path=animation_folder / Path("mse_pdm_bench.gif"),
    fps=10,
    xlabel="Y Axis",
    ylabel="Z Axis",
    title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
    vmax=0.2,
)

Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\feature_pdm\ha300\mse_pdm_bench.gif


### Potential Difference Method (PDM)

##### Testing

In [26]:
mse_pdm = mean_squared_error(
    y_true_test,
    v_x_pdm_test,
    multioutput='raw_values',
)

mean_mse_pdm = np.mean(mse_pdm)
mean_mse['PDM'] = float(mean_mse_pdm)
print(f"mean(mse(pdm)) = {mean_mse_pdm}")

mean(mse(pdm)) = 0.06624012287480881


In [27]:
i_start = 0
generate_2d_video(
    data=mse_pdm.reshape(n_x, n_y - 2, n_z),
    output_file_path=animation_folder / Path("mse_pdm.gif"),
    fps=10,
    xlabel="Y Axis",
    ylabel="Z Axis",
    title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
    vmax=0.2,
)

Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\feature_pdm\ha300\mse_pdm.gif


### Linearly Scaled PDM (cPDM)

##### Training

In [28]:
lr_c_pdm = LinearRegression(fit_intercept=False)
lr_c_pdm.fit(
    X = v_x_pdm_train.reshape(-1, 1),
    y = y_true_train.reshape(-1, 1),
)
c =  1 / lr_c_pdm.coef_[0, 0]

##### Testing

In [29]:
y_pred_c_pdm_test = v_x_pdm_test / c

mse_c_pdm = mean_squared_error(
    y_true_test,
    y_pred_c_pdm_test,
    multioutput='raw_values',
)
mean_mse_c_pdm = np.mean(mse_c_pdm)
mean_mse['c-PDM'] = float(mean_mse_c_pdm)
print(f"mean(mse(c-PDM)) = {mean_mse_c_pdm}")

mean(mse(c-PDM)) = 0.0659537695331129


In [30]:
i_start = 0
generate_2d_video(
    data=mse_c_pdm.reshape(n_x, n_y - 2, n_z),
    output_file_path=animation_folder / Path("mse_c_pdm.gif"),
    fps=10,
    xlabel="Y Axis",
    ylabel="Z Axis",
    title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
    vmax=0.2,
)

Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\feature_pdm\ha300\mse_c_pdm.gif


### Linear Regression on Feature: PDM

##### Training

In [31]:
lr_pdm = LinearRegression()
lr_pdm.fit(
    X = v_x_pdm_train.reshape(-1, 1),
    y = y_true_train.reshape(-1, 1),
);

##### Testing

In [32]:
y_pred_lr_pdm_test = lr_pdm.predict(
    v_x_pdm_test.reshape(-1, 1),
)
y_pred_lr_pdm_test = y_pred_lr_pdm_test.reshape(
    n_timesteps_test,
    n_grid_points_no_y_boundary,
)

mse_lr_pdm = mean_squared_error(
    y_true_test,
    y_pred_lr_pdm_test,
    multioutput='raw_values',
)
mean_mse_lr_pdm = np.mean(mse_lr_pdm)
mean_mse['LR-PDM'] = float(mean_mse_lr_pdm)
print(f"mse(LR-PDM) = {mean_mse_lr_pdm}")

mse(LR-PDM) = 0.0659541813700079


In [33]:
i_start = 0
generate_2d_video(
    data=mse_lr_pdm.reshape(n_x, n_y - 2, n_z),
    output_file_path=animation_folder / Path("mse_lr_pdm.gif"),
    fps=10,
    xlabel="Y Axis",
    ylabel="Z Axis",
    title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
    vmax=0.2,
)

Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\feature_pdm\ha300\mse_lr_pdm.gif


### Comparison

In [34]:
mean_mse

{'PDM Benchmark': 0.0038821085410341055,
 'PDM': 0.06624012287480881,
 'c-PDM': 0.0659537695331129,
 'LR-PDM': 0.0659541813700079}

### Appendix

##### Visualize j_y

In [35]:
j_y_train = fld_data_train[..., fld_data['labels'].index('jy')]
j_y_mean_in_time = np.mean(
    j_y_train,
    axis=0,
)
i_start = 0
generate_2d_video(
    data=j_y_mean_in_time,
    output_file_path=animation_folder / Path("j_y_mean_in_time.gif"),
    fps=10,
    xlabel="Y Axis",
    ylabel="Z Axis",
    title_pattern="X Index: " + f"{i_start} + " + " {}" if i_start != 0 else "X Index: {}",
)

Video saved to
C:\Users\hydro\Documents\PhD\nuclear_fusion_cooling\potential_probe_velocity_measuring\potential_probe_velocity_measuring\notebooks\3d\animations\feature_pdm\ha300\j_y_mean_in_time.gif
